In [1]:
import shutil
import os

In [2]:
from zipfile import ZipFile
from pathlib import Path
import glob
import json


In [3]:
results_path = "../data/results_test/*.json"
results = glob.glob(results_path)
results = [Path(p) for p in results]

In [4]:
results

[PosixPath('../data/results_test/eval_hermes-rag-beam-none-hermes-3-2-3B-base.json'),
 PosixPath('../data/results_test/eval_hermes-rag-lora-entities-beam-none-hermes-3-1-8B-entities.json'),
 PosixPath('../data/results_test/eval_hermes-lora-entities-beam-none-hermes-3-2-3B-entities.json'),
 PosixPath('../data/results_test/eval_hermes-lora-relations-beam-none-hermes-3-2-3B-relations.json'),
 PosixPath('../data/results_test/eval_hermes-lora-relations-naive-beam-none-hermes-3-2-3B-relations.json'),
 PosixPath('../data/results_test/eval_hermes-naive-beam-none-hermes-3-1-8B-base.json'),
 PosixPath('../data/results_test/eval_naive-filtered-entities.json'),
 PosixPath('../data/results_test/eval_hermes-beam-none-hermes-3-1-8B-base.json'),
 PosixPath('../data/results_test/eval_naive-entities.json'),
 PosixPath('../data/results_test/eval_hermes-rag-lora-entities-naive-beam-none-hermes-3-2-3B-entities.json'),
 PosixPath('../data/results_test/eval_hermes-beam-none-hermes-3-2-3B-base.json'),
 PosixP

In [5]:
task_ids = {
    "T611": "entities",
    "T612": "entities",
    "T621": "mention_level_relations",
    "T622": "concept_level_relations",

}
allowed_keys ={
    "T611": ["start_idx", "end_idx", "location", "text_span", "label"],
    "T612": ["start_idx", "end_idx", "location", "text_span", "label", "uri"],
    "T621": ["subject_text_span", "subject_label", "predicate", "object_text_span", "object_label"],
    "T622": ["subject_text_span", "subject_label", "subject_uri", "predicate", "object_text_span", "object_label", "object_uri"],
}
run_ids = [str(p.name).split(".")[0] for p in results]
system_id = "CHASTE"
team_id = "ToGS"
run_ids

['eval_hermes-rag-beam-none-hermes-3-2-3B-base',
 'eval_hermes-rag-lora-entities-beam-none-hermes-3-1-8B-entities',
 'eval_hermes-lora-entities-beam-none-hermes-3-2-3B-entities',
 'eval_hermes-lora-relations-beam-none-hermes-3-2-3B-relations',
 'eval_hermes-lora-relations-naive-beam-none-hermes-3-2-3B-relations',
 'eval_hermes-naive-beam-none-hermes-3-1-8B-base',
 'eval_naive-filtered-entities',
 'eval_hermes-beam-none-hermes-3-1-8B-base',
 'eval_naive-entities',
 'eval_hermes-rag-lora-entities-naive-beam-none-hermes-3-2-3B-entities',
 'eval_hermes-beam-none-hermes-3-2-3B-base',
 'eval_hermes-rag-naive-beam-none-hermes-3-1-8B-base',
 'eval_hermes-rag-lora-relations-naive-beam-none-hermes-3-1-8B-relations',
 'eval_hermes-lora-entities-beam-none-hermes-3-1-8B-entities',
 'eval_hermes-lora-relations-naive-beam-none-hermes-3-1-8B-relations',
 'eval_hermes-rag-lora-entities-beam-none-hermes-3-2-3B-entities',
 'eval_hermes-lora-entities-naive-beam-none-hermes-3-2-3B-entities',
 'eval_hermes-

In [6]:
import chevron
import json

In [10]:
staging_dir = Path("./staging")
import shutil
shutil.rmtree(staging_dir, ignore_errors=True)
zip_folders:list[Path] = []
staging_dir.mkdir(exist_ok=True)
for task_id, task_key in task_ids.items():
    for run_id, result_path in zip(run_ids, results):
        run_id_simples = run_id.replace("-", "")
        system_id_used = system_id
        desc_data = ""
        with open("./description.md", "r") as f:
            desc_data = f.read(-1)
        flags = []
        if "rag" in run_id:
            flags.append("RAG")
        if "reorder" in run_id:
            flags.append("Reordered")
        if "lora" in run_id:
            flags.append("Finetuned using LoRA")
        if "hermes-3-2-3B" in run_id:
            flags.append("Base Model: Hermes-3-2-3B")
        if "hermes-3-1-8B" in run_id:
            flags.append("Base Model: Hermes-3-1-8B")
        if "eval_naive-filtered" in run_id:
            flags.append("Run with naive approach (filtered)")
        elif "eval_naive" in run_id:
            flags.append("Run with naive approach")
            system_id_used = "NAIVE"
        rendered_desc = chevron.render(
            desc_data,
            {
                "task_id": task_id,
                "run_id": run_id,
                "system_id": system_id_used,
                "team_id": team_id,
                "flags": flags,
            },
        )
        run_data: dict[str, dict[str, any]] = None
        with open(result_path, "r") as rf:
            run_data = json.load(rf)
        stratified_res = {}
        all_empty = True
        for k, res in run_data.items():
            predictions: list[dict[str, str | int]] = res[task_key]
            predictions = [
                {k: v for k, v in p.items() if k in allowed_keys[task_id]}
                for p in predictions
            ]
            if len(predictions) > 0:
                all_empty = False
            stratified_res[k] = {task_key: predictions}

        identifier = f"{team_id}_{task_id}_{run_id_simples}_{system_id_used}"
        identifier_dir = staging_dir / identifier
        if all_empty:
            print(
                f"Warning: All predictions are empty for {identifier} on {run_id}- skipping submission generation."
            )
            continue
        identifier_dir.mkdir(exist_ok=True)
        desc_file = identifier_dir / f"{identifier}.meta"
        out_file = identifier_dir / f"{identifier}.json"
        with open(desc_file, "w") as f:
            f.write(rendered_desc)
        with open(out_file, "w") as rf:
            json.dump(stratified_res, rf)
        zip_folders.append(identifier_dir)

        # zip_path = staging_dir / f"{identifier}.zip"
        # zf = ZipFile(zip_path, "w")
        # zf.write(desc_file, f"{identifier}.md")
        # zf.write(out_file, f"{identifier}.json")

In [11]:

zip_path = staging_dir.parent / f"{team_id}_GutBrainIE_2026.zip"
with ZipFile(zip_path, "w") as zf:
    for identifier_dir in zip_folders:
        for file in identifier_dir.iterdir():
            zf.write(file, f"{identifier_dir.name}/{file.name}")